<a href="https://colab.research.google.com/github/usman-stack-322/flyrank-ml-internship-v2/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/usman-stack-322/flyrank-ml-internship-v2/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The action queue prioritizes pages that the model identifies as likely to be declining. Pages are ranked using the model's predicted probability of decline, with higher probability receiving higher priority.

Each page receives a simple reason code to make the recommendation understandable to a human reviewer:

* **HIGH_DECLINE_SIGNAL** — high predicted probability of decline.
* **MEDIUM_DECLINE_SIGNAL** — moderate predicted probability of decline.
* **LOW_DECLINE_SIGNAL** — lower predicted probability of decline.

The queue is a decision-support tool. A high priority score does not prove that refreshing the page will improve performance. Human review is required before taking action.


In [17]:
# ============================================================
# ML-10 — Setup
# Load the FlyRank dataset
# ============================================================

import os
import subprocess
import pandas as pd

REPO_URL = "https://github.com/usman-stack-322/flyrank-ml-internship-v2"
REPO_DIR = "/content/flyrank-ml-internship-v2"

# Clone the repository if it does not already exist
if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

# Move into the repository
os.chdir(REPO_DIR)

# Load dataset
DATA_PATH = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Current directory:", os.getcwd())
print("Dataset shape:", df.shape)

Current directory: /content/flyrank-ml-internship-v2
Dataset shape: (30000, 44)


In [18]:
# ============================================================
# ML-10 — Target and final model features
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# Create the target used in ML-09
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

target = "is_declining_label"

# Final 38 features used in ML-09
feature_columns = [
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "age_tier",
    "age_tier_order",
    "days_since_last_update",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "impression_tier",
    "position_tier",
]

print("Target:", target)
print("Number of features:", len(feature_columns))

Target: is_declining_label
Number of features: 38


In [19]:
# ============================================================
# ML-10 — Time-aware train/test split
# Same split design used in ML-09
# ============================================================

split_point = int(len(df) * 0.80)

train_df = df.iloc[:split_point].copy()
test_df = df.iloc[split_point:].copy()

X_train = train_df[feature_columns]
y_train = train_df[target]

X_test = test_df[feature_columns]
y_test = test_df[target]

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

Training rows: 24000
Test rows: 6000


In [20]:
# ============================================================
# ML-10 — Preprocessing and Random Forest
# Same model settings used in ML-09
# ============================================================

# Identify numeric and categorical features
numeric_features = X_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_features = [
    col for col in feature_columns
    if col not in numeric_features
]

# Numeric preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Random Forest from ML-09
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Complete pipeline
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

# Train
pipeline.fit(X_train, y_train)

print("MODEL TRAINED")
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

MODEL TRAINED
Numeric features: 29
Categorical features: 9


In [21]:
# ============================================================
# ML-10 — Preprocessing and Random Forest
# Same model settings used in ML-09
# ============================================================

# Identify numeric and categorical features
numeric_features = X_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_features = [
    col for col in feature_columns
    if col not in numeric_features
]

# Numeric preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Random Forest from ML-09
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Complete pipeline
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

# Train
pipeline.fit(X_train, y_train)

print("MODEL TRAINED")
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

MODEL TRAINED
Numeric features: 29
Categorical features: 9


In [22]:
# ============================================================
# ML-10 — Ranked actions + reason codes
# ============================================================

# Get probability that each page is declining
decline_probability = pipeline.predict_proba(X_test)[:, 1]

# Start the action queue from the test data
action_queue = test_df.copy()

# Add model score
action_queue["decline_probability"] = decline_probability

# Create human-readable reason codes
def get_reason_code(probability):
    if probability >= 0.70:
        return "HIGH_DECLINE_SIGNAL"
    elif probability >= 0.40:
        return "MEDIUM_DECLINE_SIGNAL"
    else:
        return "LOW_DECLINE_SIGNAL"

action_queue["reason_code"] = (
    action_queue["decline_probability"]
    .apply(get_reason_code)
)

# Rank highest decline signal first
action_queue = action_queue.sort_values(
    "decline_probability",
    ascending=False
).reset_index(drop=True)

# Add priority number
action_queue["priority"] = range(
    1,
    len(action_queue) + 1
)

# Columns useful for human review
queue_columns = [
    "priority",
    "decline_probability",
    "reason_code",
    "content_type",
    "content_age_days",
    "impressions_last_30d",
    "clicks_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
]

# Keep only columns that exist
queue_columns = [
    col for col in queue_columns
    if col in action_queue.columns
]

print("TOP 20 CONTENT ACTIONS")
print("-" * 60)

display(
    action_queue[queue_columns].head(20)
)

TOP 20 CONTENT ACTIONS
------------------------------------------------------------


,priority,decline_probability,reason_code,content_type,content_age_days,impressions_last_30d,clicks_last_30d,impressions_prev_30d,clicks_prev_30d
0,1,0.952306,HIGH_DECLINE_SIGNAL,keyword article,147,2583,25,6226,50
1,2,0.936754,HIGH_DECLINE_SIGNAL,keyword article,225,2489,30,7368,80
2,3,0.936563,HIGH_DECLINE_SIGNAL,keyword article,224,978,6,4110,18
3,4,0.934566,HIGH_DECLINE_SIGNAL,keyword article,225,1142,4,4354,16
4,5,0.930544,HIGH_DECLINE_SIGNAL,keyword article,224,1427,3,5184,24
5,6,0.925332,HIGH_DECLINE_SIGNAL,keyword article,147,1210,7,5089,22
6,7,0.923341,HIGH_DECLINE_SIGNAL,keyword article,132,2460,14,6298,31
7,8,0.923303,HIGH_DECLINE_SIGNAL,keyword article,224,2711,14,13031,47
8,9,0.922196,HIGH_DECLINE_SIGNAL,keyword article,132,3364,12,9172,25
9,10,0.922090,HIGH_DECLINE_SIGNAL,keyword article,224,963,3,3205,25


In [23]:
# ============================================================
# ML-10 — Action queue summary
# ============================================================

print("ACTION QUEUE SUMMARY")
print("-" * 60)

print("Total pages reviewed:", len(action_queue))

print("\nReason code counts:")
print(
    action_queue["reason_code"]
    .value_counts()
)

print("\nTop decline probability:")
print(
    round(action_queue["decline_probability"].max(), 3)
)

print("\nAverage decline probability:")
print(
    round(action_queue["decline_probability"].mean(), 3)
)

ACTION QUEUE SUMMARY
------------------------------------------------------------
Total pages reviewed: 6000

Reason code counts:
reason_code
MEDIUM_DECLINE_SIGNAL    2632
LOW_DECLINE_SIGNAL       1736
HIGH_DECLINE_SIGNAL      1632
Name: count, dtype: int64

Top decline probability:
0.952

Average decline probability:
0.517


## 2. Intended use and limits

This model is intended to support content teams in prioritizing pages for review and possible refresh. It provides a directional signal about which pages may be declining based on the observed features in this dataset.

The model should not be used to automatically publish, delete, rewrite, or refresh content. A high decline probability does not prove that a page will continue to decline or that refreshing it will improve performance.

The model is also limited to the data and validation design used here. Performance may change when the model is applied to different time periods, websites, topics, or data distributions.


In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ============================================================
# ML-10 — Intended use checks
# ============================================================

print("INTENDED USE")
print("-" * 60)
print("Use: content prioritization and decision-support")
print("Do not use: fully automated content decisions")
print("Human review: required")
print("Model result: directional signal, not causal proof")


INTENDED USE
------------------------------------------------------------
Use: content prioritization and decision-support
Do not use: fully automated content decisions
Human review: required
Model result: directional signal, not causal proof


## 3. Human review + the no-go list


Before acting on a recommendation, a human reviewer should check the page's recent performance, search intent, content quality, relevance, freshness, and whether there are known business or seasonal reasons for the observed change.

The model should never automatically publish changes, delete content, change search intent, make unsupported factual claims, or conclude that a content refresh will cause performance improvement.

The final decision remains with a human reviewer.


In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ============================================================
# ML-10 — Human review checklist
# ============================================================

review_checks = [
    "Recent performance change",
    "Search intent still matches the page",
    "Content quality and relevance",
    "Content freshness",
    "Seasonality or external demand changes",
    "Business/contextual changes",
]

no_go_actions = [
    "Automatic publishing",
    "Automatic deletion",
    "Automatic change of search intent",
    "Automatic factual claims",
    "Claiming that refresh will cause improvement",
]

print("HUMAN REVIEW CHECKS")
print("-" * 60)

for item in review_checks:
    print("[ ]", item)

print("\nNO-GO AUTOMATIONS")
print("-" * 60)

for item in no_go_actions:
    print("[X]", item)


HUMAN REVIEW CHECKS
------------------------------------------------------------
[ ] Recent performance change
[ ] Search intent still matches the page
[ ] Content quality and relevance
[ ] Content freshness
[ ] Seasonality or external demand changes
[ ] Business/contextual changes

NO-GO AUTOMATIONS
------------------------------------------------------------
[X] Automatic publishing
[X] Automatic deletion
[X] Automatic change of search intent
[X] Automatic factual claims
[X] Claiming that refresh will cause improvement


## 4. Monitoring / retrain triggers


The recommendations should be reviewed if the data distribution changes, if important features become unavailable, or if model performance falls when evaluated on a newer time period.

A retraining or validation review should be considered when the measured F1 score on a recent time-aware evaluation falls materially below the current observed result of 0.837, or when the error pattern changes substantially.

The 0.837 result is an observed validation result for this dataset, not a permanent performance guarantee.


In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ============================================================
# ML-10 — Monitoring reference
# ============================================================

current_reference_f1 = 0.837

print("MONITORING REFERENCE")
print("-" * 60)
print("Current observed time-aware F1:", current_reference_f1)
print("\nReview/retrain if:")
print("- Newer validation performance falls materially")
print("- Feature distributions change")
print("- Important input features become unavailable")
print("- Error patterns change substantially")

MONITORING REFERENCE
------------------------------------------------------------
Current observed time-aware F1: 0.837

Review/retrain if:
- Newer validation performance falls materially
- Feature distributions change
- Important input features become unavailable
- Error patterns change substantially


## 5. Exports for the paper

The ranked action queue is exported so that the paper can reuse the same evidence shown in this notebook. The exported file contains the page-level priority, model decline signal, reason code, and selected supporting performance features.


In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ============================================================
# ML-10 — Export ranked queue
# ============================================================

import os

# Create output directory
OUTPUT_DIR = "work/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Export the ranked action queue
OUTPUT_PATH = os.path.join(
    OUTPUT_DIR,
    "content_action_queue.csv"
)

action_queue[queue_columns].to_csv(
    OUTPUT_PATH,
    index=False
)

print("Export complete.")
print("Saved to:", OUTPUT_PATH)
print("Rows exported:", len(action_queue))


Export complete.
Saved to: work/outputs/content_action_queue.csv
Rows exported: 6000


In [28]:
# ============================================================
# ML-10 — Final self-check
# ============================================================

print("ML-10 SELF-CHECK")
print("-" * 60)

print("Action queue created:", not action_queue.empty)
print("Reason codes created:", "reason_code" in action_queue.columns)
print("Priority ranking created:", "priority" in action_queue.columns)
print("Export exists:", os.path.exists(OUTPUT_PATH))
print("Export path:", OUTPUT_PATH)

print("\nML-10 workflow complete.")

ML-10 SELF-CHECK
------------------------------------------------------------
Action queue created: True
Reason codes created: True
Priority ranking created: True
Export exists: True
Export path: work/outputs/content_action_queue.csv

ML-10 workflow complete.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.